In [17]:
from pathlib import Path
import re
import time
import random
import urllib.request
import urllib.error
import pandas as pd

FASTA_PATH = Path("data/raw/cullpdb_pc15.0_res0.0-2.5_noBrks_len40-10000_R0.3_Xray_d2026_01_26_chains4456.txt")

OUT_DATASET_CSV = Path("data/processed/dataset.csv")
OUT_BAD_HEADERS = Path("data/processed/unparsed_headers.txt")

STRUCT_DIR = Path("data/raw/structures_cif")
FAILED_STRUCT_LOG = Path("data/processed/failed_structures.txt")

OUT_DATASET_CSV.parent.mkdir(parents=True, exist_ok=True)
STRUCT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
def fasta_reader(path):
    header = None
    sequence = []
    with FASTA_PATH.open('r') as f:
    
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith('>'):
                if header is not None:
                    yield header, ''.join(sequence)
                header = line[1:].strip()
                sequence = []
            else:
                sequence.append(line)
        if header is not None:
            yield header, ''.join(sequence)
    return header, sequence


def extract_pdb_chain(header: str):
    token = header.split()[0].upper()

    # Must be at least 5 characters: 4 for PDB + ≥1 for chain
    if len(token) < 5:
        return None, None
    if not token[0].isdigit():
        return None, None
    if not token[:4].isalnum():
        return None, None

    pdb_id = token[:4].lower()
    chain_id = token[4:]  # can be 1+ characters

    return pdb_id, chain_id


In [3]:
rows = []
bad_headers = []

total = 0
kept = 0

for header, sequence in fasta_reader(FASTA_PATH):
    total += 1
    pdb_id, chain_id = extract_pdb_chain(header)
    if pdb_id is None:
        bad_headers.append(header)
        continue
    rows.append({
        "pdb_id": pdb_id,
        "chain_id": chain_id,
        "length": len(sequence),
        "header": header,
        "sequence": sequence
    })
    kept += 1

df = pd.DataFrame(rows)

# Save outputs
df.to_csv(OUT_DATASET_CSV, index=False)
OUT_BAD_HEADERS.write_text("\n".join(bad_headers))

print(f"Total FASTA records: {total}")
print(f"Parsed successfully: {kept}")
print(f"Unparsed headers: {total - kept}")
df.head()


Total FASTA records: 4456
Parsed successfully: 4456
Unparsed headers: 0


,pdb_id,chain_id,length,header,sequence
0,5d8v,A,83,5D8VA 92116E4FD2C44E0A 83 XRAY 0.480 0.072 ...,AAPANAVTADDPTAIALKYNQDATKSERVAAARPGLPPEEQHCANC...
1,3nir,A,46,3NIRA 919E68AF159EF722 46 XRAY 0.480 0.127 N...,TTCCPSIVARSNFNVCRLPGTPEALCATYTGCIIIPGATCPGDYAN
2,5nw3,A,54,5NW3A 7181BBB4A3E8B1A8 54 XRAY 0.590 0.135 ...,MAKWVCKICGYIYDEDAGDPDNGISPGTKFEELPDDWVCPICGAPK...
3,1ucs,A,64,1UCSA 154C6BE62D913192 64 XRAY 0.620 0.139 ...,NKASVVANQLIPINTALTLIMMKAEVVTPMGIPAEEIPKLVGMQVN...
4,3x2m,A,180,3X2MA D693A38CE0E0BC40 180 XRAY 0.640 0.122 ...,ATGGYVQQATGQASFTMYSGCGSPACGKAASGFTAAINQLAFGSAP...


In [11]:
unique_pdbs = sorted(df["pdb_id"].str.lower().unique())
len(unique_pdbs)
len(list(range(10)))
list(range(10))

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [ ]:
import requests
from pathlib import Path
import time, random

STRUCT_DIR = Path("data/raw/structures_cif")
STRUCT_DIR.mkdir(parents=True, exist_ok=True)

def download_cif(pdb_id: str, out_dir: Path, timeout: int = 30) -> bool:
    out_path = out_dir / f"{pdb_id.lower()}.cif"
    if out_path.exists() and out_path.stat().st_size > 0:
        return True

    url = f"https://files.rcsb.org/download/{pdb_id.upper()}.cif"
    try:
        r = requests.get(url, timeout=timeout)
        r.raise_for_status()
        out_path.write_bytes(r.content)
        return True
    except requests.RequestException:
        # ensure no partial file
        if out_path.exists():
            try: out_path.unlink()
            except OSError: pass
        return False

unique_pdbs = sorted(df["pdb_id"].str.lower().unique())

failed = []
for i, pdb in enumerate(unique_pdbs, 1):
    if not download_cif(pdb, STRUCT_DIR):
        failed.append(pdb)

    if i % 200 == 0 or i == len(unique_pdbs):
        print(f"[{i}/{len(unique_pdbs)}] failed={len(failed)}")

    time.sleep(0.05 + random.random() * 0.05)

failed[:20]


In [18]:
df_labels = pd.read_parquet('data/processed/rsa_sequence_dataset.parquet')

In [19]:
len_seq = []

for i in range(len(df_labels['sequence'])):
    len_seq.append(len(df_labels['sequence'][i]))

In [20]:
df_labels['len_seq'] = len_seq
df_labels

,pdb_id,chain_id,sequence,rsa,mask,len_seq
0,12as,A,AYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLS...,"[0.9056603908538818, 0.09009008854627609, 0.40...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",327
1,12as,B,AYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLS...,"[1.0, 0.07207207381725311, 0.4792899489402771,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",327
2,1a1x,A,AGEDVGAPPDHLWVHQEGIYRDEYQRTWVAVVEEETSFLRARVQQI...,"[1.0, 1.0, 0.4175257682800293, 1.0, 0.17605634...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",106
3,1a2x,A,DQQAEARSYLSEEMIAEFKAAFDMFDADGGGDISVKELGTVMRMLG...,"[1.0, 0.6111111044883728, 0.24747474491596222,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",158
4,1a2x,B,EEKRNRAITARRQHLKSVMLQIAATELEKEE,"[1.0, 0.5773195624351501, 0.8439024686813354, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",31
...,...,...,...,...,...,...
10519,9yaa,A,SMVSIALLREMFEKMVVAKNAELIGHYYDPDFVMYSDGLRQEFAEF...,"[0.7538461685180664, 0.563829779624939, 0.6267...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",119
10520,9yc2,A,GPMDRCKHVGRLRLAQDHSILNPQKWCCLECATTESVWACLKCSHV...,"[1.0, 0.6323529481887817, 0.8723404407501221, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",105
10521,9ygw,A,SASALVCLAPGSEETEAVTTIDLLVRGGIKVTTASVASDGNLAITC...,"[0.8153846263885498, 0.19811320304870605, 0.10...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",195
10522,9ygw,B,SASALVCLAPGSEETEAVTTIDLLVRGGIKVTTASVASDGNLAITC...,"[1.0, 0.17924527823925018, 0.13846154510974884...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",195


In [21]:
df_labels.to_parquet('data/processed/RSA_LABELS.parquet')

In [22]:
pd.read_parquet('data/processed/RSA_LABELS.parquet')

,pdb_id,chain_id,sequence,rsa,mask,len_seq
0,12as,A,AYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLS...,"[0.9056603908538818, 0.09009008854627609, 0.40...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",327
1,12as,B,AYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLS...,"[1.0, 0.07207207381725311, 0.4792899489402771,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",327
2,1a1x,A,AGEDVGAPPDHLWVHQEGIYRDEYQRTWVAVVEEETSFLRARVQQI...,"[1.0, 1.0, 0.4175257682800293, 1.0, 0.17605634...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",106
3,1a2x,A,DQQAEARSYLSEEMIAEFKAAFDMFDADGGGDISVKELGTVMRMLG...,"[1.0, 0.6111111044883728, 0.24747474491596222,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",158
4,1a2x,B,EEKRNRAITARRQHLKSVMLQIAATELEKEE,"[1.0, 0.5773195624351501, 0.8439024686813354, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",31
...,...,...,...,...,...,...
10519,9yaa,A,SMVSIALLREMFEKMVVAKNAELIGHYYDPDFVMYSDGLRQEFAEF...,"[0.7538461685180664, 0.563829779624939, 0.6267...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",119
10520,9yc2,A,GPMDRCKHVGRLRLAQDHSILNPQKWCCLECATTESVWACLKCSHV...,"[1.0, 0.6323529481887817, 0.8723404407501221, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",105
10521,9ygw,A,SASALVCLAPGSEETEAVTTIDLLVRGGIKVTTASVASDGNLAITC...,"[0.8153846263885498, 0.19811320304870605, 0.10...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",195
10522,9ygw,B,SASALVCLAPGSEETEAVTTIDLLVRGGIKVTTASVASDGNLAITC...,"[1.0, 0.17924527823925018, 0.13846154510974884...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",195
